In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import QuantileTransformer
import joblib

df = pd.read_csv('dataset/Teen_Mental_Health_Dataset.csv')

df.shape

(1200, 13)

La columna de academic_performace va a ser la nueva y (output varaible), en el dataset de manera original se usa depression_label como el output. Sin embargo esta columna tiene un desbalance, más del 97% de las instacias cuentan con 0. Por ese motivo hago un drop de la columna y del target.

In [2]:
X = df.drop(columns=['academic_performance', 'depression_label'])
y = df['academic_performance']

distribucion_absoluta = df['depression_label'].value_counts()
distribucion_porcentual = df['depression_label'].value_counts(normalize=True) * 100

pd.DataFrame({
    'Frecuencia Absoluta': distribucion_absoluta,
    'Proporción Porcentual (%)': distribucion_porcentual.round(2)
})

,Frecuencia Absoluta,Proporción Porcentual (%)
depression_label,,
0,1169,97.42
1,31,2.58


Para limpiar los datos primero realizo el split de los datos de train(80%) y test(20%), hacer la separación desde este punto tiene sentido ya que en posteriormente vamos a usar una técnica de escalamiento de datos, si calculamos medias numéricas o rangos utilizando el archivo completo antes del split, el set de entrenamiento tendría parámetros estadísticos del set de prueba.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print(f"Tamaño del set de train: {X_train.shape[0]} instancias")
print(f"Tamaño del set de test: {X_test.shape[0]} instancias")

Tamaño del set de train: 960 instancias
Tamaño del set de test: 240 instancias


Mostramos los tipos de datos del dataset para hacer una limpieza/homologar los tipos de datos.

In [4]:
X_train.dtypes

age                           int64
gender                          str
daily_social_media_hours    float64
platform_usage                  str
sleep_hours                 float64
screen_time_before_sleep    float64
physical_activity           float64
social_interaction_level        str
stress_level                  int64
anxiety_level                 int64
addiction_level               int64
dtype: object

La columna de nivel de interaccion social tiene string que son ordinales, por lo que vamos a hacer un ordinal mapping esta columna, para las columnas de genero y plataforma vamos a usar un one-hot enconding. AL final forzamos que los booleanos sean 0/1 y no True/False que es lo que usa Python por default.

In [5]:
# Ordinal Maping
nivel_map = {'low': 0, 'medium': 1, 'high': 2}
X_train['social_interaction_level'] = X_train['social_interaction_level'].map(nivel_map)
X_test['social_interaction_level'] = X_test['social_interaction_level'].map(nivel_map)

# One-Hot Encoding
X_train_encoded = pd.get_dummies(X_train, columns=['gender', 'platform_usage'], drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=['gender', 'platform_usage'], drop_first=True)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# Convertimos los booleanos a enteros
for col in X_train_encoded.columns:
    if X_train_encoded[col].dtype == bool or X_train_encoded[col].dtype == 'bool':
        X_train_encoded[col] = X_train_encoded[col].astype(int)
        X_test_encoded[col] = X_test_encoded[col].astype(int)

X_train_encoded.dtypes

age                           int64
daily_social_media_hours    float64
sleep_hours                 float64
screen_time_before_sleep    float64
physical_activity           float64
social_interaction_level      int64
stress_level                  int64
anxiety_level                 int64
addiction_level               int64
gender_male                   int64
platform_usage_Instagram      int64
platform_usage_TikTok         int64
dtype: object

Podemos verificar que ya los tipos de objetos son numericos int/float, lo que vamos a realizar ahora es escalar los features numericos del resto de columnas.

In [6]:
columnas_numericas = ['age', 'social_interaction_level', 'stress_level',
                     'anxiety_level', 'addiction_level']

columns_to_transform = ['daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'physical_activity']

scaler = StandardScaler()
quantile_scaler = QuantileTransformer(n_quantiles=100, output_distribution='normal')
X_train_ready = X_train_encoded.copy()
X_test_ready = X_test_encoded.copy()

X_train_ready[columnas_numericas] = scaler.fit_transform(X_train_encoded[columnas_numericas])
X_train_ready[columns_to_transform] = quantile_scaler.fit_transform(X_train_encoded[columns_to_transform])

X_test_ready[columnas_numericas] = scaler.transform(X_test_encoded[columnas_numericas])
X_test_ready[columns_to_transform] = quantile_scaler.transform(X_test_encoded[columns_to_transform])


Podemos observas que ahora todos los datos son de tipo float, menos aquellos que son datos binarios (a los que les relaizamos el one-hot encoding)

In [7]:
X_test_ready.dtypes

age                         float64
daily_social_media_hours    float64
sleep_hours                 float64
screen_time_before_sleep    float64
physical_activity           float64
social_interaction_level    float64
stress_level                float64
anxiety_level               float64
addiction_level             float64
gender_male                   int64
platform_usage_Instagram      int64
platform_usage_TikTok         int64
dtype: object

Por ultimo exportamos los datos en csv tanto para train como para test.

In [8]:
train_final_export = X_train_ready.copy()
train_final_export['target_academic_performance'] = y_train

test_final_export = X_test_ready.copy()
test_final_export['target_academic_performance'] = y_test

train_final_export.to_csv('dataset/teen_train_clean.csv', index=False)
test_final_export.to_csv('dataset/teen_test_clean.csv', index=False)

train_final_export.head(5) 

,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,gender_male,platform_usage_Instagram,platform_usage_TikTok,target_academic_performance
838,-0.456296,0.012660,0.282216,-0.229884,0.987837,0.010287,-0.504145,0.487645,1.579885,1,0,0,3.16
184,0.535877,-1.639976,0.229884,0.714776,1.690622,0.010287,-0.504145,1.535718,0.870654,0,1,0,3.06
1194,0.535877,-1.144237,-1.168949,-0.037988,-5.199338,0.010287,1.223118,-0.560428,-1.257037,1,0,0,2.65
928,-0.456296,-2.049594,0.515705,1.194396,0.486994,-1.224205,-0.849597,0.487645,-0.547807,1,0,0,2.92
461,-1.448469,0.255962,0.088734,0.870846,-0.178175,0.010287,0.877665,0.837002,0.161424,1,0,1,3.82


In [9]:
joblib.dump(scaler, 'scaler_gpa.pkl')
joblib.dump(quantile_scaler, 'quantile_scaler_gpa.pkl')

['quantile_scaler_gpa.pkl']